<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0

zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


In [2]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [3]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [4]:
import requests  # For sending HTTP requests to the SerpApi

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params, timeout=30)  # never hang the agent
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [5]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...


[1] Manila cracks prestigious list of best global travel ...
    1. The Dolomites, Milan, Italy · 2. Vancouver, Canada · 3. Beijing, China · 4. Rabat, Morocco · 5. Hull, Yorkshire, England · 6. Manila, Philippines.
    Source: https://www.postguam.com/lifestyle/world/manila-cracks-prestigious-list-of-best-global-travel-destinations/article_48f285fe-bd7c-4c84-bedf-f4381d353019.html

[2] Best US Beach Vacation Rentals for Summer 2026
    The top US beach vacation rental destinations for summer 2026 are Outer Banks NC, Gulf Shores AL, 30A/Destin FL, Cape Cod MA, Hilton Head SC, ...
    Source: https://www.houfy.com/blog/best-us-beach-vacation-rentals-for-summer-2026

[3] Travel + Leisure Names Brazil 2026 Destination of the Year
    Brazil 2026 Destination of the Year. These cities are best explored by foot, bike, or public transport.
    Source: https://www.tripadvisor.com/ShowTopic-g294280-i1045-k15482187-Travel_Leisure_Names_Brazil_2026_Destination_of_the_Year-Brazil.html

[4] Why Bhut

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [6]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [7]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        # Query the GPT-4 model with the router prompt and user input
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [8]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': "Revenue from Uber's 2021 annual report",
 'answer': "Uber's 2021 revenue"}

In [9]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': 'AI Agents are covered in OpenAI documentation',
 'answer': 'Software performing tasks autonomously'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [10]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [11]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

In [12]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", os.path.relpath(QDRANT_PATH))
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [13]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


<All keys matched successfully>


[ 1.2799684   0.4015834  -3.5162644  -0.39813134  1.5919116 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [14]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [15]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [16]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [17]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Queries financial information from an annual report
⚙️ Processing query...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**), up 57% from 2020.[1]



In [18]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Lyft's annual revenue from a financial report
⚙️ Processing query...



🤖 BOT RESPONSE:

Lyft’s revenue in 2022 was **$4.095 billion** (approximately **$4,095.1 million**), up 28% from 2021. [1]



In [19]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs
⚙️ Processing query...

Getting your response from the internet 🌐 ...


🤖 BOT RESPONSE:

Request error occurred: HTTPSConnectionPool(host='serpapi.com', port=443): Read timed out. (read timeout=30)



In [20]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?



📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API feature
⚙️ Processing query...



🤖 BOT RESPONSE:

Chat Completions lets you send a sequence of messages to a model and receive an assistant response. Each message has a `role` such as `system`, `user`, or `assistant`.

## 1. Basic request

```python
from openai import OpenAI

client = OpenAI()  # Uses the OPENAI_API_KEY environment variable

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a concise and helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain recursion in simple terms."
        }
    ]
)

answer = response.choices[0].message.content
print(answer)
```

The `system` message defines behavior, while the `user` message contains the request.

## 2. Maintain a conversation

Chat Completions is stateless, so include previous messages in each request:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "

In [21]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation
⚙️ Processing query...



🤖 BOT RESPONSE:

## Best ways to build reliable agents

### 1. Use agents only for suitable workflows
Build an agent when the system must independently execute a multi-step workflow, make decisions, use tools, recover from errors, or hand control back to a user. Simple chatbots, classifiers, and single-turn LLM calls generally do not need an agent architecture.[1]

Good candidates include:

- Customer-support resolution
- Research and report generation
- Reservation or appointment booking
- Code changes and pull requests
- Account or refund workflows

### 2. Start with the three core components
A practical agent consists of:

1. **Model** — handles reasoning and decisions  
2. **Tools** — APIs, functions, databases, search, or other systems the agent can use  
3. **Instructions** — define behavior, workflow steps, constraints, and safety rules [1]

Keep the initial design simple. Add complexity only when the workflow requires it.

### 3. Begin with the most capable model
Prototype with

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [22]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'OPENAI_QUERY', 'INTERNET_QUERY'}
bob    (finance_analyst) → {'OPENAI_QUERY', '10K_DOCUMENT_QUERY'}
carol  (unknown user)    → set()


In [23]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [24]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Uber's annual financial revenue

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [25]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Queries financial information from an annual report

✅ Access granted — processing...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [26]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [27]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation

✅ Access granted — processing...



🤖 BOT RESPONSE:

## Best ways to build reliable agents

1. **Choose an agent-worthy problem**  
   Build an agent when the workflow involves multiple steps, decisions, tool calls, or exceptions. Simple chat, classification, or single-turn generation may be better served by a conventional LLM application rather than an agent. [1]

2. **Start with the three core components**  
   Every agent should have:
   - **Model:** Handles reasoning and decisions  
   - **Tools:** APIs or functions that let it retrieve information or take action  
   - **Instructions:** Define behavior, workflow rules, and safety constraints [1]

3. **Prototype with the most capable model**  
   Establish a performance baseline using a strong model first. Then evaluate whether smaller, faster, less expensive models can handle individual steps without unacceptable accuracy loss. [1]

4. **Give the agent narrowly defined tools**  
   Tools should have clear names, inputs, outputs, and descriptions. Avoid giving one ag

'## Best ways to build reliable agents\n\n1. **Choose an agent-worthy problem**  \n   Build an agent when the workflow involves multiple steps, decisions, tool calls, or exceptions. Simple chat, classification, or single-turn generation may be better served by a conventional LLM application rather than an agent. [1]\n\n2. **Start with the three core components**  \n   Every agent should have:\n   - **Model:** Handles reasoning and decisions  \n   - **Tools:** APIs or functions that let it retrieve information or take action  \n   - **Instructions:** Define behavior, workflow rules, and safety constraints [1]\n\n3. **Prototype with the most capable model**  \n   Establish a performance baseline using a strong model first. Then evaluate whether smaller, faster, less expensive models can handle individual steps without unacceptable accuracy loss. [1]\n\n4. **Give the agent narrowly defined tools**  \n   Tools should have clear names, inputs, outputs, and descriptions. Avoid giving one age

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


## Results at a glance

All numbers below are from the saved run of this notebook (fresh kernel, top to bottom, no errors).

| | Result |
|---|---|
| **Part 1 — sub-query division** | `agentic_rag_multi()` passes all 5 table checks, including answer-level checks |
| Single question, no regression | 3 LLM calls (split + route + answer): `agentic_rag()`'s 2 plus the split, no synthesis; counted, not assumed |
| Defensive split parsing | 12/12 malformed-output cases handled offline (fences, prose, empty list, wrong key, broken JSON, `None`, …) |
| Citations across sub-answers | Renumbered globally in code (never inside code blocks), verified after synthesis, safe fallback if the model breaks them; sources listed by code |
| Grounding | Lyft 2021 now $3,208,323K (was the 2022 figure); Lyft 2024 (no filing) is reported as not found (was a remembered number with fake citations) |
| Stretch — concurrency | Answer phase 6.3 s → 3.8 s (floor 3.7 s); end to end 9–14 %, since split + compose stay sequential (Amdahl) |
| **Bonus — RBAC + semantic cache** | Self-check passes 4/4 runs; 7 extra tests pass (year guard, cross-role reuse on the shared source, role change, time-sensitive bypass, failures not cached, cache-level `PermissionError`, whole-word time check) |
| Cache design | Partitioned by **source**; permission checked before lookup and again inside the cache; threshold 0.20 measured; key-term guard for years/companies |
| Cache latency | HIT ≈ 0.9 s = router call 829 ms + lookup 40 ms (the required order routes first); MISS 1.6–6 s |
| Stretch — audit | Every request logged: user, role, route, decision, cache outcome, latency |

**Found by reading output, not by tests:** two wrong answers with correct-looking citations (see Part 1 · Step 2),
and a 42 s SerpApi stall that the original search call had no timeout for.

---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [28]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": sub_queries_prompt},
        ]
    )
  return response.choices[0].message.content


In [29]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

{"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}


### Part 1 · Step 1 — Parse the split defensively

`sub_queries()` returns **text**, not a Python list. The model may wrap its JSON in a code fence, put a sentence
around it, or return something unusable. `parse_sub_queries()` turns that text into a clean `list[str]` and, whenever
anything is off, **falls back to `[user_query]`**: the agent then answers it as one question instead of crashing.

| Model output | Result |
|---|---|
| `{"subQuestions": ["A?", "B?"]}` | `["A?", "B?"]` |
| JSON inside a ```` ```json ```` fence, or wrapped in prose | same, the JSON is found inside the text |
| `{"subQuestions": []}`, `"nope"`, `{"subQuestions": "A?"}` | fallback → `[user_query]` |
| blank items, duplicates, more than `max_parts` | cleaned: blanks and repeats dropped, list capped |

In [30]:
def parse_sub_queries(raw, user_query: str, max_parts: int = 5) -> list:
    """
    Turn the model's split (a string) into a clean list of sub-questions.
    Never raises: on any problem it falls back to [user_query].
    """
    GREY, RESET = "\033[90m", "\033[0m"

    def fallback(why: str) -> list:
        print(f"{GREY}⚠️  Split fallback ({why}) — answering as one question.{RESET}")
        return [user_query]

    if not isinstance(raw, str) or not raw.strip():
        return fallback("empty output")

    # Find the first JSON object anywhere in the text. raw_decode() parses one object
    # starting at a given position and ignores whatever follows it, so code fences
    # and surrounding prose don't matter.
    decoder = json.JSONDecoder()
    data = None
    for start in [m.start() for m in re.finditer(r"\{", raw)]:
        try:
            candidate, _ = decoder.raw_decode(raw, start)
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict) and "subQuestions" in candidate:
            data = candidate
            break
    if data is None:
        return fallback("no JSON object with 'subQuestions'")

    items = data["subQuestions"]
    if not isinstance(items, list):
        return fallback("'subQuestions' is not a list")

    # Keep real questions only: strings, not blank, no repeats (case-insensitive).
    cleaned, seen = [], set()
    for item in items:
        if isinstance(item, str) and item.strip() and item.strip().lower() not in seen:
            cleaned.append(item.strip())
            seen.add(item.strip().lower())

    if not cleaned:
        return fallback("no usable sub-questions")
    if len(cleaned) > max_parts:
        print(f"{GREY}⚠️  {len(cleaned)} sub-questions — keeping the first {max_parts}.{RESET}")
        cleaned = cleaned[:max_parts]
    return cleaned

In [31]:
# Offline tests for the parser — no API calls. Each row: (label, model output, expected result)
Q = "original question?"
cases = [
    ("clean JSON",        '{"subQuestions": ["A?", "B?"]}',                          ["A?", "B?"]),
    ("code fence",        '```json\n{"subQuestions": ["A?", "B?"]}\n```',             ["A?", "B?"]),
    ("wrapped in prose",  'Sure! Here you go: {"subQuestions": ["A?"]} Hope it helps', ["A?"]),
    ("single question",   '{"subQuestions": ["original question?"]}',                ["original question?"]),
    ("empty list",        '{"subQuestions": []}',                                    [Q]),
    ("not a list",        '{"subQuestions": "A?"}',                                  [Q]),
    ("wrong key",         '{"questions": ["A?"]}',                                   [Q]),
    ("garbage",           'I cannot split this.',                                    [Q]),
    ("broken JSON",       '{"subQuestions": ["A?", ',                                [Q]),
    ("None",              None,                                                      [Q]),
    ("blanks + dupes",    '{"subQuestions": ["A?", "  ", 42, "a?", "B?"]}',          ["A?", "B?"]),
    ("too many",          json.dumps({"subQuestions": [f"Q{i}?" for i in range(8)]}), [f"Q{i}?" for i in range(5)]),
]

for label, raw, expected in cases:
    got = parse_sub_queries(raw, Q)
    assert got == expected, f"{label}: expected {expected}, got {got}"
    print(f"✅ {label:<17} → {got}")
print(f"\nAll {len(cases)} parser cases pass.")

✅ clean JSON        → ['A?', 'B?']
✅ code fence        → ['A?', 'B?']
✅ wrapped in prose  → ['A?']
✅ single question   → ['original question?']
⚠️  Split fallback (no usable sub-questions) — answering as one question.
✅ empty list        → ['original question?']
⚠️  Split fallback ('subQuestions' is not a list) — answering as one question.
✅ not a list        → ['original question?']
⚠️  Split fallback (no JSON object with 'subQuestions') — answering as one question.
✅ wrong key         → ['original question?']
⚠️  Split fallback (no JSON object with 'subQuestions') — answering as one question.
✅ garbage           → ['original question?']
⚠️  Split fallback (no JSON object with 'subQuestions') — answering as one question.
✅ broken JSON       → ['original question?']
⚠️  Split fallback (empty output) — answering as one question.
✅ None              → ['original question?']
✅ blanks + dupes    → ['A?', 'B?']
⚠️  8 sub-questions — keeping the first 5.
✅ too many          → ['Q0?', 'Q1?', 

### Part 1 · Step 2 — Answer ONE sub-question, and keep its sources

`agentic_rag()` prints its answer and returns nothing, and `retrieve_and_response()` returns only a string.
The string contains `[1]`, but by then we've lost *which* chunk `[1]` was. To combine answers later **with citations
intact**, each sub-answer has to carry its source list along with it.

`answer_one()` returns a small dict:

| key | example |
|---|---|
| `sub_query` | `"What was Uber's revenue in 2021?"` |
| `route`, `reason` | `"10K_DOCUMENT_QUERY"`, the router's justification |
| `answer` | text with local citations `[1] [2] [3]` |
| `sources` | `["Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)", ...]` — `sources[0]` is `[1]` |
| `ok` | `False` if a step failed (API error, no hits), so the composer can say so instead of guessing |

It reuses the notebook's own pieces: `route_query()`, `get_text_embeddings()`, the Qdrant `client`,
`routes` and `get_internet_content()`. Three changes to the RAG step, each fixing a failure seen in a real run:

| Change | Failure it fixes |
|---|---|
| Chunks are handed to the model **numbered** (`[1] …`), and each number maps to a labelled source | `[1]` meant nothing once answers were combined |
| **Metadata filter**: a question naming Uber/Lyft and a year that has its own filing searches only that filing | *"Lyft 2021"* got the FY2022 filing, whose table lists 2022/2021/2020 without headers, and the model took the 2022 figure |
| **Grounded prompt**: answer only from the chunks, else reply `NOT_FOUND` (becomes `ok=False`) | *"Lyft 2024"*: no such filing exists, and the model filled in a number from memory and cited 2020/2022 filings |

In [32]:
RAG_COLLECTIONS = {"OPENAI_QUERY": "opnai_data", "10K_DOCUMENT_QUERY": "10k_data"}

# Messages the existing tools return instead of raising. An answer starting with one of
# these is a failure, not content — it must never be presented (or cached) as an answer.
FAILURE_PREFIXES = (
    "HTTP error occurred", "Request error occurred", "An unexpected error occurred",
    "No results found", "Embedding error", "Vector DB query error", "No relevant content",
    "RAG response error", "Unexpected error", "Invalid action", "Unsupported action",
    "Not found in the provided documents",
)


def describe_chunk(payload: dict, action: str) -> str:
    """Human-readable label for a retrieved chunk, built from its Qdrant metadata."""
    meta = payload.get("metadata") or {}
    if action == "10K_DOCUMENT_QUERY":
        if meta.get("company") and meta.get("fiscal_year"):
            return f"{meta['company']} 10-K, FY{meta['fiscal_year']} ({meta.get('source_file', 'filing')})"
        return f"{meta.get('company', 'Unknown company')} 10-K (fiscal year not labelled)"
    return f"OpenAI documentation — {meta.get('document_info', 'agents guide').split('/')[-1]}"


COMPANIES = {"uber": "Uber Technologies, Inc.", "lyft": "Lyft, Inc."}
FILING_YEARS = {"Uber Technologies, Inc.": {2021}, "Lyft, Inc.": {2020, 2021, 2022}}   # measured from the collection


def filing_filter(question: str):
    """Qdrant payload filter for 10-K questions that name a company (and a year with its own filing)."""
    q = question.lower()
    names = [full for short, full in COMPANIES.items() if short in q]
    if len(names) != 1:                      # no company, or a comparison of two → search everything
        return None
    conditions = [models.FieldCondition(key="metadata.company", match=models.MatchValue(value=names[0]))]
    years = [int(y) for y in re.findall(r"\b(20\d\d)\b", question)]
    if len(years) == 1 and years[0] in FILING_YEARS[names[0]]:
        conditions.append(models.FieldCondition(key="metadata.fiscal_year", match=models.MatchValue(value=years[0])))
    return models.Filter(must=conditions)


def grounded_response(user_query: str, numbered_chunks: list) -> str:
    """Like rag_formatted_response(), but may only use the chunks and must say NOT_FOUND otherwise."""
    context = "\n\n".join(numbered_chunks)
    prompt = f"""Answer the question using ONLY the numbered context below. Never use your own knowledge.

Question: {user_query}

Context:
{context}

Rules:
- Cite the chunk numbers you used, like [1][2], right after each claim.
- Financial tables often list several years side by side, sometimes without headers. State a number only if the
  context makes clear which year and company it belongs to.
- If the context does not contain the answer (for example, the requested year is not covered), reply with exactly
  NOT_FOUND: <one short sentence saying what is missing>"""
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna", messages=[{"role": "system", "content": prompt}])
    return response.choices[0].message.content.strip()


async def retrieve_with_sources(user_query: str, action: str):
    """Embed → search (filtered for 10-K questions) → grounded answer. Returns (answer, sources)."""
    query_vector = get_text_embeddings(user_query)
    query_filter = filing_filter(user_query) if action == "10K_DOCUMENT_QUERY" else None
    hits = await client.query_points(collection_name=RAG_COLLECTIONS[action], query=query_vector,
                                     query_filter=query_filter, limit=3)
    if not hits.points:
        return "No relevant content found in the database.", []

    sources = [describe_chunk(p.payload, action) for p in hits.points]
    numbered = [f"[{i}] {p.payload['content']}" for i, p in enumerate(hits.points, start=1)]
    answer = grounded_response(user_query, numbered)
    if answer.startswith("NOT_FOUND"):
        return "Not found in the provided documents: " + answer.split(":", 1)[-1].strip(), sources
    return answer, sources


def internet_sources(search_text: str) -> list:
    """Pull '[n] title … Source: url' blocks out of get_internet_content()'s text, in [n] order."""
    found = re.findall(r"^\[(\d+)\] (.*?)\n.*?Source: (\S+)", search_text, re.MULTILINE | re.DOTALL)
    by_number = {int(n): f"{title.strip()} — {url}" for n, title, url in found}
    return [by_number[n] for n in sorted(by_number)]


def execute_route(query: str, action: str):
    """Run the tool for an already-decided route. Returns (answer, sources, ok)."""
    try:
        if action in RAG_COLLECTIONS:
            answer, sources = asyncio.run(retrieve_with_sources(query, action))
        elif action == "INTERNET_QUERY":
            answer = routes[action](query, action)                # get_internet_content()
            sources = internet_sources(answer)
        else:
            answer, sources = f"Unsupported action: {action}", []
    except Exception as err:
        answer, sources = f"Unexpected error: {err}", []
    return answer, sources, bool(answer) and not answer.startswith(FAILURE_PREFIXES)


def answer_one(sub_query: str) -> dict:
    """Route one sub-question, run the matching tool, and return answer + sources."""
    GREY, RESET = "\033[90m", "\033[0m"

    decision = route_query(sub_query)
    action, reason = decision.get("action"), decision.get("reason", "")
    print(f"{GREY}   ↳ {sub_query!r} → {action}  ({reason}){RESET}")

    answer, sources, ok = execute_route(sub_query, action)
    return {"sub_query": sub_query, "route": action, "reason": reason,
            "answer": answer, "sources": sources, "ok": ok}

In [33]:
# Try it on the two halves of the question that failed earlier
for q in ["What was Uber's revenue in 2021?", "What are the newest LLMs?"]:
    r = answer_one(q)
    print(f"\nroute={r['route']}  ok={r['ok']}")
    print("answer :", r["answer"][:300].replace("\n", " "), "…" if len(r["answer"]) > 300 else "")
    for i, s in enumerate(r["sources"], start=1):
        print(f"  [{i}] {s}")
    print("-" * 70)

   ↳ "What was Uber's revenue in 2021?" → 10K_DOCUMENT_QUERY  (Asks for Uber's annual financial revenue)



route=10K_DOCUMENT_QUERY  ok=True
answer : Uber's revenue in 2021 was **$17.455 billion**. [2][3] 
  [1] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)
  [2] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)
  [3] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)
----------------------------------------------------------------------


   ↳ 'What are the newest LLMs?' → INTERNET_QUERY  (Requires current information about newly released language models)
Getting your response from the internet 🌐 ...



route=INTERNET_QUERY  ok=True
answer : [1] Exploring the Best Open Source LLMs of 2024     Learn which open-source LLMs are the best in terms of adaptability, manageability, and quality (what are the baseline metrics provided)     Source: https://dagshub.com/blog/best-open-source-llms/  [2] The Best Open-Source Coding LLMs in 2026 (Ranke …
  [1] Exploring the Best Open Source LLMs of 2024 — https://dagshub.com/blog/best-open-source-llms/
  [2] The Best Open-Source Coding LLMs in 2026 (Ranked by ... — https://pub.towardsai.net/the-best-open-source-coding-llms-in-2026-ranked-by-real-developer-use-cases-f8e5a25ed251
  [3] llama2 may not be as smart as newer LLMs, but it does ... — https://www.reddit.com/r/LocalLLaMA/comments/1oe99n3/llama2_may_not_be_as_smart_as_newer_llms_but_it/
  [4] Shivam Bharadwaj's Post — https://www.linkedin.com/posts/shivam-bharadwaj-ai_the-newest-llms-are-getting-larger-but-the-activity-7499438161643749376-BB6b
  [5] Introduce new LLMs into copilot (Students do

### Part 1 · Step 3 — Compose one answer, keeping every citation correct

Each sub-answer numbers its own sources from `[1]`. Glued together, two different documents would both be `[1]`
and a reader would believe both claims came from the same place. So composing has three stages:

1. **Renumber (code, not the model).** Each part's citations are shifted by the number of sources before it:
   part 1 keeps `[1]–[3]`, part 2's `[1]` becomes `[4]`, and so on. Brackets inside code (`choices[0]`) are left alone.
2. **Synthesise (one LLM call).** The model writes one answer from the renumbered parts and must copy the
   citation markers exactly.
3. **Verify (code, not the model).** Any citation number that doesn't exist, or a successful part whose citations
   all vanished, means the synthesis can't be trusted. We then fall back to a plain per-question layout that keeps
   the renumbered citations. The **Sources** list is always built by code, never written by the model.

In [34]:
CITATION = re.compile(r"\[(\d+)\]")
CODE_SPAN = re.compile(r"```.*?```|`[^`\n]*`", re.DOTALL)   # fenced blocks and `inline code`


def outside_code(text: str, fn) -> str:
    """Apply fn to the prose parts of text only; code blocks pass through untouched."""
    pieces, last = [], 0
    for m in CODE_SPAN.finditer(text):
        pieces += [fn(text[last:m.start()]), m.group()]
        last = m.end()
    return "".join(pieces + [fn(text[last:])])


def citations_in(text: str) -> set:
    found = set()
    outside_code(text, lambda prose: found.update(int(n) for n in CITATION.findall(prose)) or prose)
    return found


def renumber(parts: list):
    """Shift each part's local [n] into one global numbering. Returns (new parts, global source list)."""
    all_sources, renumbered = [], []
    for part in parts:
        offset, count = len(all_sources), len(part["sources"])

        def shift(prose, offset=offset, count=count):
            # only numbers that point at a real source of THIS part are shifted
            return CITATION.sub(lambda m: f"[{int(m.group(1)) + offset}]" if 1 <= int(m.group(1)) <= count
                                else m.group(), prose)

        renumbered.append({**part, "answer": outside_code(part["answer"], shift),
                           "citations": [offset + i for i in range(1, count + 1)]})
        all_sources += part["sources"]
    return renumbered, all_sources


def check_citations(final: str, parts: list, n_sources: int) -> list:
    """Return a list of problems with the composed answer's citations (empty list = trustworthy)."""
    cited, problems = citations_in(final), []
    invented = sorted(n for n in cited if not 1 <= n <= n_sources)
    if invented:
        problems.append(f"cites sources that don't exist: {invented}")
    for p in parts:
        used_in_part = citations_in(p["answer"]) & set(p["citations"])
        if p["ok"] and used_in_part and not (used_in_part & cited):
            problems.append(f"dropped every citation for: {p['sub_query']!r}")
    return problems


def synthesise(user_query: str, parts: list) -> str:
    blocks = []
    for i, p in enumerate(parts, start=1):
        status = "ANSWERED" if p["ok"] else "FAILED — say this part could not be answered; do not guess"
        blocks.append(f"### Part {i}: {p['sub_query']}\nStatus: {status}\nRoute: {p['route']}\n{p['answer']}")
    prompt = f"""You combine partial answers into ONE answer for the user.

User question: {user_query}

Partial answers (already researched):
{chr(10).join(blocks)}

Rules:
- Use ONLY facts from the partial answers. Add nothing from your own knowledge.
- Keep citation markers EXACTLY as written ([4] stays [4]) and put each right after the claim it supports.
  Never invent a new number. Web-search parts list raw results as [n] Title / snippet: cite a result only
  if its snippet actually supports what you write.
- Cover every part, in order. For a FAILED part, say plainly it could not be answered.
- Do not add a sources list; it is appended automatically."""
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna", messages=[{"role": "system", "content": prompt}])
    return response.choices[0].message.content.strip()


def plain_layout(parts: list) -> str:
    """Deterministic fallback: each question with its own (renumbered) answer."""
    return "\n\n".join(
        f"**{p['sub_query']}**\n" + (p["answer"] if p["ok"] else "⚠️ Could not be answered: " + p["answer"])
        for p in parts)


def sources_block(final: str, all_sources: list) -> str:
    cited = sorted(n for n in citations_in(final) if 1 <= n <= len(all_sources))
    if not cited:
        return ""
    return "\n\nSources:\n" + "\n".join(f"[{n}] {all_sources[n - 1]}" for n in cited)


def compose(user_query: str, parts: list) -> str:
    """Renumber → synthesise → verify → append sources. Falls back to plain_layout if verification fails."""
    GREY, YELLOW, RESET = "\033[90m", "\033[93m", "\033[0m"
    parts, all_sources = renumber(parts)
    try:
        final = synthesise(user_query, parts)
        problems = check_citations(final, parts, len(all_sources))
    except Exception as err:
        final, problems = "", [f"synthesis call failed: {err}"]

    if problems:
        print(f"{YELLOW}⚠️  Synthesis rejected ({'; '.join(problems)}) — using the plain layout.{RESET}")
        final = plain_layout(parts)
    else:
        print(f"{GREY}🔎 Citation check passed: {sorted(citations_in(final))} all point at real sources.{RESET}")
    return final + sources_block(final, all_sources)

In [35]:
# Offline tests for renumbering and the citation check — no API calls.
fake = [
    {"sub_query": "Uber 2021 revenue?", "ok": True, "route": "10K_DOCUMENT_QUERY",
     "answer": "Uber made $17.5B [1][2].", "sources": ["uber-a", "uber-b", "lyft-c"]},
    {"sub_query": "How to call the API?", "ok": True, "route": "OPENAI_QUERY",
     "answer": "Read `response.choices[0]` [1]. See [2].\n```python\nmsgs[1]\n```", "sources": ["docs-a", "docs-b"]},
]
parts, srcs = renumber(fake)
assert parts[0]["answer"] == "Uber made $17.5B [1][2]."                       # first part unchanged
assert parts[1]["answer"].startswith("Read `response.choices[0]` [4]. See [5].")  # shifted by 3
assert "msgs[1]" in parts[1]["answer"]                                          # code untouched
assert srcs == ["uber-a", "uber-b", "lyft-c", "docs-a", "docs-b"]
print("✅ renumber: part 2 shifted by 3, code brackets untouched")

assert check_citations("Uber made $17.5B [1]. Use `choices[0]` [4].", parts, 5) == []
print("✅ check: good answer passes (brackets inside `code` are not citations)")
assert "don't exist" in check_citations("Uber made $17.5B [1]. API [9].", parts, 5)[0]
print("✅ check: invented [9] caught")
assert "dropped" in check_citations("Uber made $17.5B [1]. The API is easy.", parts, 5)[0]
print("✅ check: a part whose citations all vanished is caught")
print("\n" + sources_block("Uber [1], API [4].", srcs))

✅ renumber: part 2 shifted by 3, code brackets untouched
✅ check: good answer passes (brackets inside `code` are not citations)
✅ check: invented [9] caught
✅ check: a part whose citations all vanished is caught



Sources:
[1] uber-a
[4] docs-a


In [36]:
# ── Part 1: your implementation ──────────────────────────────────────────────

async def answer_all_concurrently(questions: list) -> list:
    """Stretch: run answer_one() for every sub-query at the same time, each in its own thread.
    Almost all of answer_one() is waiting on OpenAI / SerpApi / Qdrant, so the waits overlap.
    gather() keeps the results in the same order as the questions, so citation numbering is unchanged."""
    return await asyncio.gather(*(asyncio.to_thread(answer_one, q) for q in questions))


def agentic_rag_multi(user_query: str, return_details: bool = False, parallel: bool = False):
    """
    Split a compound query, run each sub-query through the agentic pipeline,
    and synthesise one final answer.

    Args:
        user_query (str): Possibly compound question.
        return_details (bool): If True, return (answer, parts) so tests can inspect routes.
        parallel (bool): Stretch — answer the sub-queries concurrently instead of one after another.

    Returns:
        str: A single composed answer covering every sub-question.
    """
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # 1. Split — call sub_queries(), parse the JSON defensively
    try:
        raw_split = sub_queries(user_query)
    except Exception as err:                      # API down → still answer, as one question
        raw_split = None
        print(f"{GREY}⚠️  Split call failed: {err}{RESET}")
    questions = parse_sub_queries(raw_split, user_query)
    print(f"{GREY}🧩 {len(questions)} sub-quer{'y' if len(questions) == 1 else 'ies'}:{RESET}")

    # 2. Route + answer each sub-query independently (each may land on a different source)
    if parallel and len(questions) > 1:
        parts = asyncio.run(answer_all_concurrently(questions))
    else:
        parts = [answer_one(q) for q in questions]

    # 3. Compose one final answer, keeping citations
    if len(parts) == 1:
        # Single question: no synthesis call, so this costs exactly what agentic_rag() costs + the split.
        part = parts[0]
        final = part["answer"]
        if part["route"] in RAG_COLLECTIONS:      # web results already carry their own "Source:" links
            final += sources_block(final, part["sources"])
    else:
        final = compose(user_query, parts)

    print(f"\n{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n\n{final}\n")
    return (final, parts) if return_details else final

### Part 1 · Step 4 — Check against the table

Each test below runs the real pipeline and **asserts** the expected behaviour from the assignment table: number of
sub-queries, and whether the routes are the same or different. It also counts OpenAI calls, to prove the
single-question case costs only one call more than `agentic_rag()` (the split) and no synthesis.

| Query | Expected |
|---|---|
| `what was uber revenue in 2021?` | 1 sub-query, 1 route, **no synthesis call** |
| `what was lyft revenue in 2021 and what was uber revenue in 2021` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `what was uber's 2021 revenue and what are the newest LLMs?` | 2 sub-queries, **different** routes |
| 3-part: 10-K + OpenAI docs + web | 3 sub-queries, 3 different routes |
| `…Uber's revenue in 2021 and what was Lyft's in 2024?` (the notebook's own example) | Lyft 2024 isn't in the filings, so that part must come back `ok=False`, never a number |

On top of routing, two **answer checks**: a Lyft 2021 answer must contain the FY2021 figure ($3,208,323 thousand), and a Lyft 2024 part must be `ok=False`.

In [37]:
import time
from collections import Counter

# Count every OpenAI call (split, routing, generation, synthesis) by wrapping the client method.
_real_create = openaiclient.chat.completions.create
llm_calls = Counter()
def _counting_create(*args, **kwargs):
    llm_calls["n"] += 1
    return _real_create(*args, **kwargs)
openaiclient.chat.completions.create = _counting_create

tests = [
    ("what was uber revenue in 2021?",                                    1, "single"),
    ("what was lyft revenue in 2021 and what was uber revenue in 2021",  2, "same"),
    ("what was uber's 2021 revenue and what are the newest LLMs?",       2, "different"),
    ("What was Lyft's revenue in 2022, how do I add guardrails to an OpenAI agent, "
     "and what are the most popular open-source LLMs?",                   3, "different"),
    ("What was Uber's revenue in 2021 and what was Lyft's in 2024?",      2, "same"),
]

summary = []
try:
    for query, n_expected, routes_expected in tests:
        print("=" * 90)
        llm_calls.clear(); t0 = time.perf_counter()
        answer, parts = agentic_rag_multi(query, return_details=True)
        seconds = time.perf_counter() - t0
        routes_seen = [p["route"] for p in parts]

        assert len(parts) == n_expected, f"expected {n_expected} sub-queries, got {len(parts)}"
        if routes_expected == "same":
            assert len(set(routes_seen)) == 1, f"expected one route, got {routes_seen}"
        if routes_expected == "different":
            assert len(set(routes_seen)) == len(routes_seen), f"expected different routes, got {routes_seen}"
        if routes_expected == "single":
            # agentic_rag() = route + generate = 2 calls for a 10-K question; we may add only the split
            assert llm_calls["n"] == 3, f"single question used {llm_calls['n']} LLM calls, expected 3"
        answers = {p["sub_query"].lower(): p for p in parts}
        for sq, p in answers.items():
            if "lyft" in sq and "2021" in sq:          # FY2021 revenue is $3,208,323 thousand
                assert "3,208" in p["answer"] or "3.2" in p["answer"], f"wrong Lyft 2021 figure: {p['answer'][:120]}"
            if "lyft" in sq and "2024" in sq:          # no FY2024 filing exists in the collection
                assert not p["ok"], f"invented a Lyft 2024 figure: {p['answer'][:120]}"
        summary.append((query[:60], len(parts), " + ".join(r.replace("_QUERY", "") for r in routes_seen),
                        llm_calls["n"], f"{seconds:.1f}s"))
finally:
    openaiclient.chat.completions.create = _real_create      # always restore the real client

print("=" * 90)
print(f"\n{'query':<62}{'subs':>5}  {'routes':<34}{'LLM calls':>9}{'time':>8}")
for row in summary:
    print(f"{row[0]:<62}{row[1]:>5}  {row[2]:<34}{row[3]:>9}{row[4]:>8}")
print(f"\n✅ All {len(summary)} table checks pass.")

👤 User Query: what was uber revenue in 2021?



🧩 1 sub-query:


   ↳ "What was Uber's revenue in 2021?" → 10K_DOCUMENT_QUERY  (Asks about Uber's annual financial results)



🤖 BOT RESPONSE:

Uber's revenue in 2021 was **$17.455 billion**. [2]

Sources:
[2] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021



🧩 2 sub-queries:


...

🔎 Citation check passed: [1, 2, 4, 5, 8, 11] all point at real sources.

🤖 BOT RESPONSE:

## 1. Lyft’s 2022 revenue

Lyft reported **$4,095,135,000** in revenue in 2022. [1][2]

## 2. Adding guardrails to an OpenAI agent

Using the Agents SDK:

1. Identify risks for your use case, especially data privacy and content safety. Add additional guardrails as real-world edge cases emerge, while balancing security and user experience. [4]

2. Define a guardrail function or agent that enforces a policy, such as jailbreak prevention, relevance validation, keyword filtering, blocklist enforcement, or safety classification. [5]

3. Attach it through the agent’s `input_guardrails` configuration:

```python
customer_support_agent = Agent(
    name="Customer support agent",
    instructions=(
        "You are a customer support agent. "
        "You help customers with their questions."
    ),
    input_guardrails=[
        Guardrail(guardrail_function=churn_detection_tripwire),
    ],
)
```

[4][5]


🧩 2 sub-queries:


   ↳ "What was Uber's revenue in 2021?" → 10K_DOCUMENT_QUERY  (Asks about Uber's annual financial revenue)


   ↳ "What was Lyft's revenue in 2024?" → 10K_DOCUMENT_QUERY  (Queries Lyft's annual financial results)


🔎 Citation check passed: [1, 2] all point at real sources.

🤖 BOT RESPONSE:

Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1][2]

Lyft's revenue in 2024 could not be answered because it was not found in the provided documents.

Sources:
[1] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)
[2] Uber Technologies, Inc. 10-K, FY2021 (uber_10k_2021.htm)


query                                                          subs  routes                            LLM calls    time
what was uber revenue in 2021?                                    1  10K_DOCUMENT                              3    2.2s
what was lyft revenue in 2021 and what was uber revenue in 2      2  10K_DOCUMENT + 10K_DOCUMENT               6    6.4s
what was uber's 2021 revenue and what are the newest LLMs?        2  10K_DOCUMENT + INTERNET                   5    6.7s
What was Lyft's revenue in 2022, how do I add guardrails to       3  10K_DOCUMENT + OPENAI + INTERNET          7   14.

### Part 1 · Stretch — sequential vs. concurrent

`agentic_rag_multi(q, parallel=True)` answers the sub-queries with `asyncio.gather` + `asyncio.to_thread`
instead of a `for` loop. Nothing else changes: same `answer_one`, same composer. The benchmark times the same
compound queries both ways (2 runs each, alternating so neither mode always goes first) and checks both modes
route the sub-queries identically.

**What the numbers mean** (measured 2026-09-27; exact seconds vary run to run):

```
 sequential:  split ─ [ sub-query 1 ][ sub-query 2 ][ sub-query 3 ] ─ compose
 concurrent:  split ─ [ sub-query 1 ]                               ─ compose
                      [ sub-query 2 ]
                      [ sub-query 3 ]
```
- In the **answer phase**, concurrency works almost perfectly: in the saved run, 3 sub-queries took 6.3 s one after
  another and 3.8 s together, right at the 3.7 s floor set by the slowest sub-query.
- **End to end** the saving is smaller (9–14 % in the saved run, up to ~25 % in others) because the split call and the compose call run once
  in both modes and are about half the total time. Speeding up one phase can only save that phase's share:
  this is Amdahl's law. The next win would be a smaller, faster model for the compose step, not more concurrency.
- Both modes route every sub-query identically, and `gather()` returns results in question order, so citation
  numbering is the same in both.

In [38]:
import contextlib, io, statistics, time

bench_queries = [
    "what was uber's 2021 revenue and what are the newest LLMs?",
    "What was Lyft's revenue in 2022, how do I add guardrails to an OpenAI agent, "
    "and what are the most popular open-source LLMs?",
]

def timed(query, parallel):
    with contextlib.redirect_stdout(io.StringIO()):          # keep the benchmark output short
        t0 = time.perf_counter()
        _, parts = agentic_rag_multi(query, return_details=True, parallel=parallel)
    return time.perf_counter() - t0, [p["route"] for p in parts]

rows = []
for query in bench_queries:
    seq, par = [], []
    for run in range(2):
        order = [False, True] if run == 0 else [True, False]
        for parallel in order:
            seconds, routes_seen = timed(query, parallel)
            (par if parallel else seq).append((seconds, routes_seen))
    same_routes = all(sorted(r) == sorted(seq[0][1]) for _, r in seq + par)
    s, p = statistics.mean(t for t, _ in seq), statistics.mean(t for t, _ in par)
    rows.append((query[:58], len(seq[0][1]), s, p, same_routes))

print(f"{'query':<60}{'subs':>5}{'sequential':>12}{'concurrent':>12}{'saved':>8}  same routes")
for q, n, s, p, same in rows:
    print(f"{q:<60}{n:>5}{s:>11.1f}s{p:>11.1f}s{(1 - p / s):>8.0%}  {'✅' if same else '⚠️'}")


# Where does the time go? Time ONLY the answer phase, on a fixed list of sub-queries
# (no split call, no compose call) — the part that parallel mode actually changes.
fixed = ["What was Lyft's revenue in 2022?", "How do I add guardrails to an OpenAI agent?",
         "What are the most popular open-source LLMs?"]
with contextlib.redirect_stdout(io.StringIO()):
    t0 = time.perf_counter(); seq_parts = [answer_one(q) for q in fixed];                   t_seq = time.perf_counter() - t0
    t0 = time.perf_counter(); par_parts = asyncio.run(answer_all_concurrently(fixed));       t_par = time.perf_counter() - t0
    one = []
    for q in fixed:
        t0 = time.perf_counter(); answer_one(q); one.append(time.perf_counter() - t0)

print(f"\nAnswer phase only (3 sub-queries): sequential {t_seq:.1f}s → concurrent {t_par:.1f}s "
      f"({1 - t_par / t_seq:.0%} saved)")
print("Each sub-query alone: " + ", ".join(f"'{' '.join(q.split()[:4])}…' {t:.1f}s" for q, t in zip(fixed, one)))
print(f"Best possible concurrent time ≈ the slowest one: {max(one):.1f}s")

query                                                        subs  sequential  concurrent   saved  same routes
what was uber's 2021 revenue and what are the newest LLMs?      2        6.2s        5.6s      9%  ✅
What was Lyft's revenue in 2022, how do I add guardrails t      3       12.2s       10.5s     14%  ✅



Answer phase only (3 sub-queries): sequential 6.3s → concurrent 3.8s (41% saved)
Each sub-query alone: 'What was Lyft's revenue…' 1.9s, 'How do I add…' 3.7s, 'What are the most…' 0.8s
Best possible concurrent time ≈ the slowest one: 3.7s


---

## Bonus (optional) — RBAC with a semantic cache

*Not graded on its own — but this is rehearsal for graded work. **ARGUS**, the full-stack
assignment for this module, requires a real caching layer and a cost panel showing spend per
100 sources with and without it. Build the cache here, on a system you already understand,
and you'll be extending it in ARGUS rather than meeting it for the first time under a deadline.*

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


In [39]:
# ── Bonus: your implementation ───────────────────────────────────────────────
# Reuse USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS from Section 6.

# NOTE: import sentence_transformers BEFORE faiss. Both ship their own OpenMP runtime and on macOS
# the kernel dies silently the other way round (see rag_helpers.py).
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import sys, time, os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless fork warning
if IN_COLAB:                                      # rag_helpers.py lives in the cloned module folder
    sys.path.insert(0, f"/content/multi-agent-course/{_MODULE}")
from rag_helpers import SemanticCaching          # only for its TIME_SENSITIVE_KEYWORDS list

# ── Never cache time-sensitive questions ─────────────────────────────────────
# Same keyword idea as SemanticCaching.is_time_sensitive() in notebook 003, with two changes:
#  • whole-word matching, so "know" no longer matches "now" (a false positive only skips the cache,
#    but a cache that is skipped for no reason is a cache that doesn't work);
#  • "newest"/"new" added: "what are the newest LLMs?" is exactly the kind of answer that goes stale.
_TIME_WORDS = SemanticCaching.TIME_SENSITIVE_KEYWORDS + ["newest", "new", "trending"]
_TIME_SENSITIVE = re.compile(r"\b(" + "|".join(re.escape(w) for w in _TIME_WORDS) + r")\b", re.IGNORECASE)

def is_time_sensitive(question: str) -> bool:
    return bool(_TIME_SENSITIVE.search(question))

# ── Key-term guard ───────────────────────────────────────────────────────────
# Measured below: "uber revenue in 2020?" is 0.084 from "uber revenue in 2021?", as close as a real
# paraphrase (0.09). No threshold can tell them apart, because embeddings capture the topic, and the
# topic is the same. So a hit also requires the same numbers and the same company names.
ENTITY_WORDS = set(COMPANIES) | {"openai"}

def key_terms(question: str) -> frozenset:
    q = question.lower()
    numbers = set(re.findall(r"\d+(?:\.\d+)?", q))
    entities = {w for w in ENTITY_WORDS if re.search(rf"\b{w}\b", q)}
    return frozenset(numbers | entities)


_ENCODER = None   # loaded once and shared, so building a new cache (as the self-check does) is cheap

class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice (partitioned vs. tagged): PARTITIONED, by knowledge SOURCE (the route label),
    not by role. One FAISS index per source: OPENAI_QUERY, 10K_DOCUMENT_QUERY, INTERNET_QUERY.

      • Why it can't leak: the only way into a partition is the RBAC check for that same source, which
        secure_agentic_rag_cached() runs BEFORE the lookup, and check()/add() re-run it themselves
        (defence in depth). An answer in the 10-K partition was built only from 10-K data, so anyone
        who may reach that partition may read it. alice can never search it; there is nothing to filter.
      • Why per source, not per role (the hint's trade-off): alice and bob share OPENAI_QUERY. Per-role
        partitions would compute and store that answer twice; per-source partitions compute it once and
        serve both. The cost: this is only correct while permissions are granted per WHOLE source. If a
        role were limited to part of a source (say, Lyft filings only, via a Qdrant payload filter), the
        partition key would have to include that scope too, e.g. ("10K_DOCUMENT_QUERY", "lyft-only").
      • Role changes: permissions are checked on every request against the user's CURRENT role, so if
        bob moves from finance to engineering his old 10-K entries become unreachable to him at once,
        with no cache purge needed. They stay available to users who still hold that permission, which
        is correct: the data didn't become secret, bob's access changed.
      • Why not tagged: one shared index where each hit is checked against a role tag works too, but a
        forgotten tag check is a silent leak. With partitions, the unsafe lookup is impossible to write.

    Two more rules: a hit also needs the same key terms (numbers and company names, see key_terms), and
    the threshold is measured on this notebook's questions (cell below), not guessed.
    """

    def __init__(self, threshold: float = 0.20):
        global _ENCODER
        if _ENCODER is None:
            _ENCODER = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
        self.encoder = _ENCODER
        self.threshold = threshold                 # squared L2 on normalised vectors; lower = stricter
        self.partitions = {}                       # source → {"index": faiss index, "entries": [dict, ...]}

    def _authorize(self, user_id: str, source: str):
        if not has_access(user_id, source):
            raise PermissionError(f"{user_id!r} may not use the {source} cache partition")

    def _partition(self, source: str) -> dict:
        return self.partitions.setdefault(source, {"index": faiss.IndexFlatL2(768), "entries": []})

    def embed(self, question: str):
        return self.encoder.encode([question], normalize_embeddings=True).astype(np.float32)

    def check(self, user_id: str, question: str, source: str):
        """Return (hit: bool, answer: str | None, embedding, similarity | None)."""
        self._authorize(user_id, source)
        embedding = self.embed(question)
        part = self.partitions.get(source)
        if not part or part["index"].ntotal == 0:
            return False, None, embedding, None

        # Look at the 3 nearest, not just 1: the closest may be a different year of the same question.
        distances, rows = part["index"].search(embedding, min(3, part["index"].ntotal))
        terms = key_terms(question)
        for distance, row in zip(distances[0], rows[0]):
            if row == -1 or distance > self.threshold:
                break
            entry = part["entries"][row]
            if entry["terms"] == terms:
                return True, entry["answer"], embedding, float(1 - distance)
        return False, None, embedding, None

    def add(self, user_id: str, question: str, answer: str, embedding, source: str):
        """Store an answer in the partition of the source it was built from."""
        self._authorize(user_id, source)
        part = self._partition(source)
        part["index"].add(embedding)
        part["entries"].append({"question": question, "answer": answer, "terms": key_terms(question),
                                "added_by": user_id, "added_at": time.time()})

    def size(self) -> dict:
        return {source: len(p["entries"]) for source, p in self.partitions.items()}


# ── Audit trail (stretch) ────────────────────────────────────────────────────
AUDIT_LOG = []

def print_audit_log(last: int = None):
    rows = AUDIT_LOG[-last:] if last else AUDIT_LOG
    print(f"{'#':>3}  {'user':<6}{'role':<16}{'route':<20}{'decision':<9}{'cache':<24}{'ms':>7}  query")
    for i, r in enumerate(rows, start=1):
        print(f"{i:>3}  {r['user']:<6}{r['role'] or '—':<16}{(r['route'] or '—').replace('_QUERY', ''):<20}"
              f"{r['decision']:<9}{r['cache']:<24}{r['ms']:>7.0f}  {r['query'][:48]}")


def secure_agentic_rag_cached(user_id: str, user_query: str, cache) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware semantic cache.

    Order: identity → route → permission → cache → pipeline.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
              plus "route", "cache" (what happened with the cache) and "similarity" for inspection.
    """
    CYAN, GREY, RED, GREEN, YELLOW, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[93m", "\033[1m", "\033[0m")
    t0 = time.perf_counter()
    role = USERS.get(user_id)

    def finish(answer, status, route=None, cache_note="—", similarity=None):
        ms = (time.perf_counter() - t0) * 1000
        AUDIT_LOG.append({"user": user_id, "role": role, "query": user_query, "route": route,
                          "decision": "DENIED" if status == "DENIED" else "ALLOWED",
                          "cache": cache_note, "ms": ms})
        colour = {"HIT": GREEN, "MISS": YELLOW, "DENIED": RED}[status]
        print(f"{BOLD}{CYAN}👤 {user_id}{RESET} ({role or 'UNKNOWN'}) · {user_query!r}\n"
              f"   {colour}{status}{RESET}{GREY} · route={route or '—'} · cache={cache_note} · {ms:.0f} ms{RESET}")
        return {"answer": answer, "status": status, "role": role, "route": route,
                "cache": cache_note, "similarity": similarity}

    # 1. unknown user → DENIED (no embedding, no cache, no LLM)
    if role is None:
        return finish(f"🚫 Access denied: unknown user '{user_id}'.", "DENIED", cache_note="not consulted")

    # 2. route the query → which source does this need?
    action = route_query(user_query).get("action")

    # 3. role not allowed → DENIED (still no cache lookup; a denial is never cached)
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        return finish(f"🚫 Access denied: your role ('{role}') does not have permission to query {source}.",
                      "DENIED", route=action, cache_note="not consulted")

    # 4a. time-sensitive → answer live, never read or write the cache
    if is_time_sensitive(user_query):
        answer, sources, ok = execute_route(user_query, action)
        if action in RAG_COLLECTIONS:
            answer += sources_block(answer, sources)
        return finish(answer, "MISS", route=action, cache_note="bypassed (time-sensitive)")

    # 4b. cache lookup → HIT: return the stored answer
    hit, cached_answer, embedding, similarity = cache.check(user_id, user_query, action)
    if hit:
        return finish(cached_answer, "HIT", route=action, cache_note=f"hit (sim {similarity:.2f})",
                      similarity=similarity)

    # 4c. MISS → run the pipeline, store (only if it actually worked), return
    answer, sources, ok = execute_route(user_query, action)
    if action in RAG_COLLECTIONS:
        answer += sources_block(answer, sources)
    if ok:
        cache.add(user_id, user_query, answer, embedding, action)
        note = "miss → stored"
    else:
        note = "miss → NOT stored (failed)"      # a cached error would fail every later user too
    return finish(answer, "MISS", route=action, cache_note=note)

### Bonus · Measuring the threshold (don't guess it)

Squared L2 distance between normalised Nomic embeddings, as `faiss.IndexFlatL2` reports it. We want a gap:
paraphrases below the threshold, different questions above. The run showed one pair **no threshold can fix**:
a different *year* is as close as a paraphrase. The key-term guard handles that one.

In [40]:
probe = RoleAwareSemanticCache()
base = "what was uber revenue in 2021?"
pairs = [
    (base, "how much revenue did Uber make in 2021?",                       "HIT"),
    (base, "What was Uber's total revenue for fiscal year 2021?",            "HIT"),
    (base, "what was uber revenue in 2020?",                                 "MISS"),
    (base, "what was lyft revenue in 2021?",                                 "MISS"),
    (base, "what was uber net loss in 2021?",                                "MISS"),
    ("how do I build an agent with the OpenAI Agents SDK?",
     "how can I create an agent using the OpenAI Agents SDK?",               "HIT"),
    ("how do I build an agent with the OpenAI Agents SDK?",
     "how do I add guardrails to an OpenAI agent?",                          "MISS"),
]
print(f"threshold = {probe.threshold}\n")
print(f"{'dist':>6}  {'distance says':<14}{'+ key terms':<13}{'want':<6} question")
for a, b, want in pairs:
    va, vb = probe.embed(a)[0], probe.embed(b)[0]
    d = float(np.sum((va - vb) ** 2))
    by_distance = "HIT" if d <= probe.threshold else "miss"
    final = "HIT" if by_distance == "HIT" and key_terms(a) == key_terms(b) else "miss"
    mark = "✅" if final.upper() == want else "❌"
    print(f"{d:>6.3f}  {by_distance:<14}{final:<13}{want:<6} {b}  {mark}")

<All keys matched successfully>


threshold = 0.2

  dist  distance says + key terms  want   question


 0.090  HIT           HIT          HIT    how much revenue did Uber make in 2021?  ✅
 0.087  HIT           HIT          HIT    What was Uber's total revenue for fiscal year 2021?  ✅
 0.084  HIT           miss         MISS   what was uber revenue in 2020?  ✅
 0.302  miss          miss         MISS   what was lyft revenue in 2021?  ✅
 0.341  miss          miss         MISS   what was uber net loss in 2021?  ✅
 0.084  HIT           HIT          HIT    how can I create an agent using the OpenAI Agents SDK?  ✅
 0.570  miss          miss         MISS   how do I add guardrails to an OpenAI agent?  ✅


In [41]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


run_self_check()

👤 bob (finance_analyst) · 'what was uber revenue in 2021?'
   MISS · route=10K_DOCUMENT_QUERY · cache=miss → stored · 1619 ms


👤 bob (finance_analyst) · 'what was uber revenue in 2021?'
   HIT · route=10K_DOCUMENT_QUERY · cache=hit (sim 1.00) · 1054 ms


👤 alice (engineer) · 'what was uber revenue in 2021?'
   DENIED · route=10K_DOCUMENT_QUERY · cache=not consulted · 748 ms


👤 alice (engineer) · 'how much revenue did Uber make in 2021?'
   DENIED · route=10K_DOCUMENT_QUERY · cache=not consulted · 812 ms
👤 carol (UNKNOWN) · 'how do I build an agent with the OpenAI Agents SDK?'
   DENIED · route=— · cache=not consulted · 0 ms


👤 alice (engineer) · 'how do I build an agent with the OpenAI Agents SDK?'
   MISS · route=OPENAI_QUERY · cache=miss → stored · 5096 ms


👤 alice (engineer) · 'how do I build an agent with the OpenAI Agents SDK?'
   HIT · route=OPENAI_QUERY · cache=hit (sim 1.00) · 767 ms
✅ All checks passed — cache is fast and does not leak across roles.


### Bonus · Beyond the self-check

The self-check passes once. The router is an LLM, so one pass could be luck: it runs **3 more times** below.
Then the cases the self-check doesn't cover:

| # | Test | Expected |
|---|---|---|
| 1 | bob asks the **2020** version after 2021 is cached | MISS (key-term guard), never 2021's answer |
| 2 | bob asks the OpenAI question alice already cached | **HIT**: shared source, computed once for both roles |
| 3 | bob's role changes to engineer, then back | DENIED while engineer, HIT again after |
| 4 | a time-sensitive question, asked twice | MISS both times, cache size unchanged |
| 5 | the pipeline fails (simulated) | MISS twice, the error is **not** cached |
| 6 | calling the cache directly for a forbidden partition | `PermissionError`: the cache defends itself |
| 7 | `is_time_sensitive("how do I know …")` | `False`: whole-word match |

It ends with the **audit table** (stretch): who asked what, route, decision, cache outcome and latency.

**Why a HIT takes ~1 s and not ~40 ms.** The required order is *route → permission → cache*, so every request,
hits included, pays for one router LLM call before the cache is consulted. The measurement in the cell shows the
split: the lookup itself is milliseconds, the router is the rest. A hit still skips retrieval and generation
(1.5–5 s, 2 LLM calls). The faster alternative would be to look up the cache *before* routing, across every
partition the user's role may read (the "tagged by permitted source set" design). It would give ~40 ms hits,
but the cache would then decide the source instead of the router, and it breaks the order this exercise requires.
That trade-off is worth revisiting in ARGUS, where cache latency is a measured number.

In [42]:
# Robustness: the self-check 3 more times (fresh cache each time)
for i in range(3):
    print(f"\n──── self-check repeat {i + 1}/3 ────")
    run_self_check()

AUDIT_LOG.clear()
cache = RoleAwareSemanticCache()
q_fin, q_doc = "what was uber revenue in 2021?", "how do I build an agent with the OpenAI Agents SDK?"
print("\n──── setup: bob caches the 2021 question, alice caches the OpenAI question ────")
assert secure_agentic_rag_cached("bob", q_fin, cache)["status"] == "MISS"
assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"

print("\n──── 1. different year must not hit ────")
r = secure_agentic_rag_cached("bob", "what was uber revenue in 2020?", cache)
assert r["status"] == "MISS", f"served the 2021 answer for 2020: {r['status']}"

print("\n──── 2. shared source is shared across roles ────")
r = secure_agentic_rag_cached("bob", q_doc, cache)
assert r["status"] == "HIT", f"expected cross-role HIT on the shared source, got {r['status']}"

print("\n──── 3. role change ────")
USERS["bob"] = "engineer"
try:
    assert secure_agentic_rag_cached("bob", q_fin, cache)["status"] == "DENIED"
finally:
    USERS["bob"] = "finance_analyst"
assert secure_agentic_rag_cached("bob", q_fin, cache)["status"] == "HIT"

print("\n──── 4. time-sensitive is never cached ────")
before = cache.size()
for _ in range(2):
    r = secure_agentic_rag_cached("alice", "what are the newest open-source LLMs this week?", cache)
    assert r["status"] == "MISS" and r["cache"].startswith("bypassed"), r
assert cache.size() == before, f"time-sensitive answer was stored: {before} → {cache.size()}"

print("\n──── 5. failures are never cached ────")
_real_execute = execute_route
execute_route = lambda query, action: ("HTTP error occurred: 503 Service Unavailable", [], False)
try:
    for _ in range(2):
        r = secure_agentic_rag_cached("bob", "what was lyft revenue in 2022?", cache)
        assert r["status"] == "MISS" and "NOT stored" in r["cache"], r
finally:
    execute_route = _real_execute

print("\n──── 6. the cache refuses a forbidden partition on its own ────")
try:
    cache.check("alice", q_fin, "10K_DOCUMENT_QUERY")
    raise AssertionError("cache served a forbidden partition")
except PermissionError as err:
    print("   PermissionError:", err)

print("\n──── where a HIT's time goes ────")
t0 = time.perf_counter(); route_query(q_fin);                                        t_route = time.perf_counter() - t0
t0 = time.perf_counter(); hit, *_ = cache.check("bob", q_fin, "10K_DOCUMENT_QUERY");  t_lookup = time.perf_counter() - t0
assert hit
print(f"   router LLM call: {t_route * 1000:.0f} ms · cache lookup (embed + FAISS): {t_lookup * 1000:.0f} ms")

print("\n──── 7. whole-word time check ────")
assert not is_time_sensitive("how do I know which model to use?")
assert is_time_sensitive("what is the current stock price of Uber?")

print(f"\n✅ All extra checks pass. Cache contents per source: {cache.size()}\n")
print_audit_log()


──── self-check repeat 1/3 ────


👤 bob (finance_analyst) · 'what was uber revenue in 2021?'
   MISS · route=10K_DOCUMENT_QUERY · cache=miss → stored · 1710 ms


👤 bob (finance_analyst) · 'what was uber revenue in 2021?'
   HIT · route=10K_DOCUMENT_QUERY · cache=hit (sim 1.00) · 802 ms


👤 alice (engineer) · 'what was uber revenue in 2021?'
   DENIED · route=10K_DOCUMENT_QUERY · cache=not consulted · 714 ms


👤 alice (engineer) · 'how much revenue did Uber make in 2021?'
   DENIED · route=10K_DOCUMENT_QUERY · cache=not consulted · 710 ms
👤 carol (UNKNOWN) · 'how do I build an agent with the OpenAI Agents SDK?'
   DENIED · route=— · cache=not consulted · 0 ms


...

Getting your response from the internet 🌐 ...
👤 alice (engineer) · 'what are the newest open-source LLMs this week?'
   MISS · route=INTERNET_QUERY · cache=bypassed (time-sensitive) · 924 ms


Getting your response from the internet 🌐 ...
👤 alice (engineer) · 'what are the newest open-source LLMs this week?'
   MISS · route=INTERNET_QUERY · cache=bypassed (time-sensitive) · 950 ms

──── 5. failures are never cached ────


👤 bob (finance_analyst) · 'what was lyft revenue in 2022?'
   MISS · route=10K_DOCUMENT_QUERY · cache=miss → NOT stored (failed) · 1066 ms


👤 bob (finance_analyst) · 'what was lyft revenue in 2022?'
   MISS · route=10K_DOCUMENT_QUERY · cache=miss → NOT stored (failed) · 984 ms

──── 6. the cache refuses a forbidden partition on its own ────
   PermissionError: 'alice' may not use the 10K_DOCUMENT_QUERY cache partition

──── where a HIT's time goes ────


   router LLM call: 829 ms · cache lookup (embed + FAISS): 40 ms

──── 7. whole-word time check ────

✅ All extra checks pass. Cache contents per source: {'10K_DOCUMENT_QUERY': 2, 'OPENAI_QUERY': 1}

  #  user  role            route               decision cache                        ms  query
  1  bob   finance_analyst 10K_DOCUMENT        ALLOWED  miss → stored              1939  what was uber revenue in 2021?
  2  alice engineer        OPENAI              ALLOWED  miss → stored              5874  how do I build an agent with the OpenAI Agents S
  3  bob   finance_analyst 10K_DOCUMENT        ALLOWED  miss → stored              1840  what was uber revenue in 2020?
  4  bob   finance_analyst OPENAI              ALLOWED  hit (sim 1.00)              977  how do I build an agent with the OpenAI Agents S
  5  bob   engineer        10K_DOCUMENT        DENIED   not consulted               899  what was uber revenue in 2021?
  6  bob   finance_analyst 10K_DOCUMENT        ALLOWED  hit (sim 1.00